# rl18_grid

> Phase 18, Part B (`specs/multiagent/phase18/`): RL of clones of one model with team reward on the colony board against the plain board (both with two holders per clue). Arms and runs (`RL18_ARMS`, `RL18_RUNS`), the configuration (`rl18Config`: Phase 17's frozen recipe, mix and liars, the board through `world`), the speed check's configuration, the training-window rule (`maxLenRule`), the hours projection, the evaluation suite (`colonySuite`) and `RL18_PLAN`. Runs are claimed across Colab accounts with Phase 17's `claim`.

In [ ]:
#| default_exp rl18_grid

In [ ]:
#| hide
from nbdev.showdoc import *

## Arms, runs, configurations

In [ ]:
#| export
import math

from explRL.clue_grid import BOARD, CLUE_SETTINGS, clueCell
from explRL.coop_grid import RL_PLAN
from explRL.rl17_grid import BASE17, SIZES, RL17Config

RL18_ARMS = dict(c18team=dict(world=(('colony', True), ('r', 2)), scheme='team'),
                 p18team=dict(world=(('r', 2),), scheme='team'))
RL18_SEEDS = dict(c18team=5)          # Part B lite (user, 2026-10-21): 5 colony runs, the least for an exact sign-flip p < 0.05;
                                      # p18team (plain board) is kept defined for a later full Part B, not run
RL18_RUNS = [(a, s) for s in range(1, 9) for a in RL18_ARMS if s <= RL18_SEEDS.get(a, 0)]   # 5 runs
RL18_STEPS = 30                       # half Phase 17's 60
RL18_MIX = dict(mixN=((4, 0.5), (8, 0.5)), liarShare=0.25)
EVAL_SEEDS = list(range(900, 905))
EVAL_SIZES = (8, 32)
BOARDS18 = dict(colony=dict(colony=True), plain={})


def rl18Config(arm, seed, model, out, recipe, steps, maxLen):
    "A Part B run: RL_PLAN v2 + BASE17 (with FIX17) + Phase 17's recipe + the mix + the arm; trainSeed = seed - 1; a checkpoint every step."
    return RL17Config(model, out, **{**RL_PLAN['rl'], **BASE17, **recipe, **RL18_MIX, **RL18_ARMS[arm], 'steps': steps,
                                     'trainSeed': seed - 1, 'ckptEvery': 1, 'maxLen': maxLen})


def speedConfig(model, out, recipe):
    "The speed check: c18team, seed 1, 3 steps, a 4,096-token window (to measure the longest prompt)."
    return rl18Config('c18team', 1, model, out, recipe, 3, 4096)


def maxLenRule(longest, floor=3072, step=512):
    "The training window: the longest measured prompt plus reply rounded up to a multiple of 512, at least FIX17's 3,072."
    return max(floor, int(math.ceil(longest / step) * step))


def projectHours18(stepSeconds, steps=RL18_STEPS, seeds=RL18_SEEDS):
    "GPU hours from the speed check's seconds per step (every run plays the same mix): per run and in all."
    run = stepSeconds * steps / 3600
    return dict(perRun=run, total=sum(seeds.values()) * run)

## Evaluation suite

In [ ]:
#| export
def colonySuite(models, seeds=EVAL_SEEDS, sizes=EVAL_SIZES, boards=('colony',), liars=(True,)):
    """Per model, the given boards (r = 2; Part B lite: the colony board), sizes (N 8 and 32) and liar settings (with N // 4
    independent liars), 40 turns, notes memory; levels '<model>|<board>n<N>' and '<model>|<board>n<N>liar' (study colonyEval,
    runner 'local'). The full Part B suite: boards=('colony', 'plain'), sizes=SIZES, liars=(False, True), 10 seeds."""
    cells = []
    for m in models:
        for b in boards:
            w = {**{k: CLUE_SETTINGS[k] for k in BOARD}, 'r': 2, **BOARDS18[b]}
            for N in sizes:
                for liar in liars:
                    kw = dict(nLiar=N // 4) if liar else {}
                    cells += [clueCell('colonyEval', f'{m}|{b}n{N}' + 'liar' * liar, N, sd, 'local', channel='blackboard',
                                       model=f'local:{m}', memory='notes', world=w, **kw) for sd in seeds]
    return cells

## Plan

In [ ]:
#| export
RL18_PLAN = dict(
    version=2, frozen=False, recipe=None, steps=RL18_STEPS, maxLen=None, projectedHours=None, userConfirmed=False,
    amendment=dict(date='2026-10-21', by='user', what='Part B lite: 5 c18team runs of 30 steps, no p18team arm, a smaller '
                   'evaluation, H18-B2 the only primary', why='the full design projected about 150 GPU hours'),
    arms={a: dict(v, world=[list(x) for x in v['world']]) for a, v in RL18_ARMS.items()}, seeds=RL18_SEEDS,
    runs=[list(r) for r in RL18_RUNS], mix=dict(mixN=[list(x) for x in RL18_MIX['mixN']], liarShare=RL18_MIX['liarShare']),
    common='BASE17 (credit settings with scheme overridden by the arm: team; FIX17), RL_PLAN v2',
    speedCheck='c18team seed 1, 3 steps, maxLen 4096: seconds per step, longest prompt plus reply, cut prompts',
    maxLenRule='longest measured prompt plus reply rounded up to a multiple of 512, at least 3072; any cut prompt in a main run is reported',
    evaluation='colonySuite: warm start and the 5 adapters; colony board (r 2), N 8 and 32 with N // 4 liars, seeds 900-904, 40 turns',
    hypotheses=dict(
        H18_B2='primary: per c18team run, mean specIndex - specNull over its colonyn8liar boards; one-sided exact sign-flip > 0 '
               '(5 runs: p >= 1/32)',
        H18_B3='secondary: per c18team run, honestPooled at colonyn32liar - the warm start\'s; one-sided sign-flip, reported',
        H18_B4='secondary: freeRiding and misreport at colonyn8liar against the warm start, bootstrap over runs',
        trainTrend='secondary: per run, specIndex - specNull of the training rollouts (turns/step-*.jsonl) per step and its slope '
                   'over steps; a rising curve at step 30 marks a null H18-B2 as "not within 30 steps"',
        H18_B1='deferred to a full Part B (p18team runs on the plain board): not tested'),
    holm=['H18_B2'], alpha=0.05, nboot=2000, exploratory=True,
    gates='speed check: RL17_PLAN frozen with a recipe and Phase 18 Stage 1 done; main runs: also a stability check of Phase 17\'s '
          'main runs (no halts or collapse under the recipe) and the user\'s confirmation of the projected hours')

## Tests

In [ ]:
import json
from dataclasses import asdict
from explRL.rl18_grid import *
from explRL.rl17_grid import PILOT17, SIZES, configHash17, RL17Config
from explRL.llm_runs import LLMConfig

assert RL18_RUNS == [('c18team', s) for s in range(1, 6)] and RL18_PLAN['steps'] == RL18_STEPS == 30
m = '/content/drive/MyDrive/explRL_rl/warm_model'
c = rl18Config('c18team', 3, m, 'o', PILOT17['p17a'], 200, 3584)
assert (c.scheme, c.trainSeed, c.steps, c.maxLen, c.valueOnly, c.tokenClip, c.liarShare) == ('team', 2, 200, 3584, True, True, 0.25)
assert dict(c.world) == dict(colony=True, r=2) and c.mixN == ((4, 0.5), (8, 0.5))
p = rl18Config('p18team', 3, m, 'o', PILOT17['p17a'], 200, 3584)
assert dict(p.world) == dict(r=2) and configHash17(p) != configHash17(c)
assert configHash17(RL17Config(**{**asdict(c), 'ckptEvery': 9})) == configHash17(c)
s = speedConfig(m, 'o', PILOT17['p17a'])
assert (s.steps, s.maxLen, s.trainSeed) == (3, 4096, 0)
assert maxLenRule(2200) == 3072 and maxLenRule(3100) == 3584 and maxLenRule(3584) == 3584
h = projectHours18(720)
assert abs(h['perRun'] - 6) < 1e-9 and abs(h['total'] - 5 * 6) < 1e-9
cells = colonySuite(['warm', 'c18team_s1'])
assert len(cells) == 2 * 1 * 2 * 1 * 5 and {x['level'] for x in cells} == {f'{m}|colonyn{N}liar' for m in ('warm', 'c18team_s1') for N in (8, 32)}
assert len(colonySuite(['warm'], seeds=range(10), sizes=SIZES, boards=('colony', 'plain'), liars=(False, True))) == 2 * 4 * 2 * 10
for x in cells:
    cfg = LLMConfig.fromDict(x['cfg'])
    assert cfg.world['r'] == 2 and cfg.world.get('colony', False) == ('|colony' in x['level']) and cfg.turns == 40
    assert cfg.nLiar == (cfg.N // 4 if x['level'].endswith('liar') else 0)
json.dumps(RL18_PLAN)
# AC-9: two simulated accounts split the runs and then the evaluation models by claims (Phase 17's claim)
import tempfile
from pathlib import Path
from explRL.rl17_grid import claim
with tempfile.TemporaryDirectory() as d:
    runs, got = [f'{a}_s{s}' for a, s in RL18_RUNS], {}
    for n in runs:                                   # account a takes a run, b skips it, alternating who arrives first
        first, second = ('a', 'b') if len(got) % 2 == 0 else ('b', 'a')
        assert claim(Path(d) / 'runs_p18' / n, first) and not claim(Path(d) / 'runs_p18' / n, second)
        got[n] = first
    assert set(got.values()) == {'a', 'b'} and len(got) == 5
    models = ['warm'] + runs
    ev = {m: next(h for h in ('a', 'b') if claim(Path(d) / 'eval_p18' / 'claims' / m, h)) for m in models}
    assert all(not claim(Path(d) / 'eval_p18' / 'claims' / m, 'b' if h == 'a' else 'a') for m, h in ev.items())
    assert claim(Path(d) / 'eval_p18' / 'claims' / '_pack', 'b') and not claim(Path(d) / 'eval_p18' / 'claims' / '_pack', 'a')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()